# EDA de enfermedades y síntomas

Dataset: `DiseaseAndSymptoms.csv` (ver `../data/raw/README.md`). Este notebook documenta **qué decisiones de limpieza se toman y por qué**, y valida los dos catálogos que usa el ETL:

- `../mappings/label_map.csv`: las enfermedades que cubrirá el modelo y su código CIE-10.
- `../mappings/sintomas.csv`: el vocabulario de síntomas, con su traducción al español.

No modifica nada en `raw/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("../data/raw")
MAPPINGS = Path("../mappings")

cabecera = pd.read_csv(RAW / "DiseaseAndSymptoms.csv", nrows=0).columns
sintomas = [c for c in cabecera if c != "diseases"]
# int8: el CSV pesa ~190 MB; con int64 ocuparía más de 700 MB en memoria
df = pd.read_csv(RAW / "DiseaseAndSymptoms.csv", dtype={c: "int8" for c in sintomas})
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
enfermedades = df['diseases']
enfermedades.head()

## 1. Estructura y valores

Qué esperamos: una columna `diseases` y 377 columnas 0/1, sin nulos.

In [ ]:
print("Nulos:", int(df.isna().sum().sum()))
print("Valores en las columnas de síntomas:", int(df[sintomas].min().min()), "a", int(df[sintomas].max().max()))
print("Enfermedades:", df["diseases"].nunique(), "| síntomas:", len(sintomas))
print("Columnas con nombre repetido en el CSV original:", [c for c in sintomas if c.endswith(".1")])

**Hallazgo:** el CSV original trae dos columnas llamadas `regurgitation`; pandas renombra la segunda como `regurgitation.1`. Se conservan como dos síntomas distintos (ver sección 7).

## 2. Escritura de los nombres

¿Hay espacios sobrantes, mayúsculas o dobles espacios?

In [ ]:
normalizar = lambda t: " ".join(str(t).lower().split())
print("Síntomas sin normalizar:", [c for c in sintomas if c != normalizar(c)])
print("Enfermedades sin normalizar:", int((df["diseases"] != df["diseases"].map(normalizar)).sum()), "filas")

**Hallazgo:** los nombres ya vienen limpios, pero `transform.py` aplica `normalizar` de todos modos para que el vocabulario sea idéntico al de la BD y al que usará `features.py`.

## 3. Duplicados

Filas idénticas (misma enfermedad y mismos síntomas). Si no se quitan **antes** de dividir en train/val/test, la misma fila cae en ambos conjuntos y la precisión sale inflada.

In [ ]:
n_dup = int(df.duplicated().sum())
print(f"Filas duplicadas exactas: {n_dup:,} ({n_dup / len(df):.1%})")
dd = df.drop_duplicates().reset_index(drop=True)
print(f"Filas tras deduplicar: {len(dd):,}")

## 4. Balance de clases

In [ ]:
antes = df["diseases"].value_counts()
despues = dd["diseases"].value_counts()
print(pd.DataFrame({"antes": antes.describe(), "después de deduplicar": despues.describe()}).round(1))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
ax[0].hist(antes, bins=40); ax[0].set_title("Filas por enfermedad (original)")
ax[1].hist(despues, bins=40); ax[1].set_title("Casos únicos por enfermedad")
for a in ax: a.set_xlabel("filas"); 
ax[0].set_ylabel("enfermedades"); plt.tight_layout()

In [ ]:
umbrales = [10, 30, 50, 100, 200, 300, 500]
pd.Series({f">= {u} casos únicos": int((despues >= u).sum()) for u in umbrales}, name="enfermedades")

**Hallazgo:** de 773 enfermedades, muchas quedan con muy pocos casos únicos (la mediana baja de 168 a 77). Con menos de ~200 casos no hay datos suficientes para entrenar y evaluar. Eso obliga a trabajar con un **subconjunto**.

## 5. Sesgo del dataset: los casos únicos dependen del número de síntomas posibles

Cada enfermedad tiene entre 2 y 12 síntomas posibles y el dataset genera combinaciones de ellos. Cuantos más síntomas posibles, más combinaciones distintas.

In [ ]:
posibles = dd.groupby("diseases")[sintomas].max().sum(axis=1)
tabla = (pd.DataFrame({"sintomas_posibles": posibles, "casos_unicos": despues})
         .groupby("sintomas_posibles")["casos_unicos"].agg(enfermedades="count", mediana_casos="median"))
tabla

**Hallazgo:** el número de casos de una enfermedad refleja cuántos síntomas tiene, no qué tan frecuente es en la realidad. Las enfermedades con 12 síntomas aportan ~1200 casos y las de 9, ~300. El modelo verá cierto desbalance: conviene usar pesos por clase o muestreo estratificado, y no interpretar las probabilidades como prevalencias.

## 6. Contradicciones: el mismo conjunto de síntomas en varias enfermedades

In [ ]:
llave = pd.Series([fila.tobytes() for fila in dd[sintomas].values])
n_enf = dd["diseases"].groupby(llave).nunique()
ambiguas = llave.map(n_enf) > 1
print(f"Conjuntos de síntomas distintos: {len(n_enf):,}")
print(f"Conjuntos compartidos por más de una enfermedad: {(n_enf > 1).sum():,}")
print(f"Filas afectadas: {ambiguas.sum():,} ({ambiguas.mean():.1%})")

**Decisión:** **no se eliminan.** Son ambigüedad clínica real (un mismo cuadro puede ser varias enfermedades) y el modelo debe devolver varias opciones con su probabilidad, que es lo que ya define `schemas/prediccion.py`. Lo que sí se elimina son las filas idénticas (sección 3).

## 7. La columna `regurgitation` duplicada

In [ ]:
for col in ["regurgitation", "regurgitation.1"]:
    print(col, "→", dd[dd[col] == 1]["diseases"].value_counts().head(5).to_dict())

**Hallazgo:** ambas aparecen sobre todo en cuadros digestivos (hernia de hiato, colecistitis, gastritis…) y no hay forma de distinguirlas clínicamente con estos datos. Se conservan como dos síntomas distintos (`regurgitación` y `regurgitación (variante 2)`) para no perder información del dataset; la interfaz puede mostrar solo una de las dos.

## 8. Subconjunto de enfermedades

**Criterios** (todos deben cumplirse):

1. Al menos **200 casos únicos** tras deduplicar (265 enfermedades lo cumplen).
2. Se puede mapear a **una categoría CIE-10 de 3 caracteres** presente en `ICD-10.csv`, sin repetir códigos.
3. Es un cuadro que se sospecha desde un listado de síntomas en consulta general. Se excluyen lesiones, intoxicaciones, cuadros obstétricos, entradas vagas (`nose disorder`, `skin disorder`…) y los que requieren imagen o laboratorio.
4. No es casi indistinguible de otra enfermedad ya incluida (Jaccard de síntomas ≥ 0.75).

Las enfermedades que cumplían 1–3 pero se excluyeron en el último ajuste:

| Enfermedad | Motivo |
|---|---|
| erectile dysfunction | el código N52 no existe en `ICD-10.csv` |
| post-traumatic stress disorder (ptsd) | 91% de síntomas en común con depression |
| noninfectious gastroenteritis | 85% en común con infectious gastroenteritis |
| pyelonephritis | 83% en común con kidney stone |
| chronic obstructive pulmonary disease (copd) | 77% en común con acute bronchitis; requiere espirometría |
| gallstone | 77% en común con cholecystitis |
| macular degeneration | 75% en común con cataract |
| sickle cell crisis, pulmonary hypertension, molluscum contagiosum, labyrinthitis, ankylosing spondylitis | especializadas |
| intracerebral hemorrhage, pleural effusion, uterine fibroids, ovarian cyst | requieren imagen |
| sebaceous cyst, pyogenic skin infection, eustachian tube dysfunction | diagnóstico inespecífico |
| sepsis | emergencia que requiere laboratorio |

> La selección y los códigos CIE-10 son una **propuesta** y deben ser revisados por alguien con formación médica. La columna `tipo_mapeo` indica si el código coincide con la enfermedad (`exacto`) o si es una categoría más amplia que la agrupa (`amplio`).

In [ ]:
lm = pd.read_csv(MAPPINGS / "label_map.csv")
icd = pd.read_csv(RAW / "ICD-10.csv")
icd3 = icd[icd["id"].astype(str).str.match(r"^[A-Za-z]\d{2}$")].set_index("id")["text"]

chequeos = {
    "todas existen en el dataset": lm["enfermedad_en"].isin(despues.index).all(),
    "todas tienen >= 200 casos únicos": (lm["enfermedad_en"].map(despues) >= 200).all(),
    "el código CIE-10 existe": lm["cie10_codigo"].isin(icd3.index).all(),
    "un código por enfermedad (sin repetidos)": lm["cie10_codigo"].is_unique,
    "la descripción coincide con ICD-10.csv": (lm["cie10_codigo"].map(icd3) == lm["cie10_descripcion"]).all(),
}
for nombre, ok in chequeos.items():
    print("OK  " if ok else "FALLA", nombre)
assert all(chequeos.values())
print("\nEnfermedades en el subconjunto:", len(lm))

In [ ]:
lm.groupby("sistema").agg(enfermedades=("enfermedad_en", "count"), casos_unicos=("casos_unicos", "sum")).sort_values("enfermedades", ascending=False)

In [ ]:
print(lm["tipo_mapeo"].value_counts().to_string())
print(f"\nCasos únicos por enfermedad: min {lm.casos_unicos.min()}, mediana {int(lm.casos_unicos.median())}, max {lm.casos_unicos.max()}")

In [ ]:
sub = dd[dd["diseases"].isin(lm["enfermedad_en"])]
U = sub.groupby("diseases")[sintomas].max().astype("float32")
inter = U.values @ U.values.T
tam = U.values.sum(axis=1)
jac = inter / (tam[:, None] + tam[None, :] - inter)
np.fill_diagonal(jac, 0)
nombres = list(U.index)
pares = sorted(((jac[i, j], nombres[i], nombres[j]) for i in range(len(nombres)) for j in range(i + 1, len(nombres))), reverse=True)[:8]
pd.DataFrame(pares, columns=["jaccard", "enfermedad_a", "enfermedad_b"]).round(2)

Los pares más parecidos del subconjunto quedan por debajo de 0.75. El modelo podrá distinguirlos, aunque mostrará dudas entre cuadros emparentados (p. ej. espondilosis y estenosis espinal).

## 9. Vocabulario de síntomas

In [ ]:
sv = pd.read_csv(MAPPINGS / "sintomas.csv")
chequeos = {
    "377 síntomas": len(sv) == len(sintomas) == 377,
    "mismo orden que las columnas del CSV": (sv["nombre_en"].tolist() == sintomas),
    "orden = 0..376": (sv["orden"].tolist() == list(range(377))),
    "sin nombres en español repetidos": sv["nombre_es"].is_unique,
    "sin traducciones vacías": sv["nombre_es"].notna().all(),
}
for nombre, ok in chequeos.items():
    print("OK  " if ok else "FALLA", nombre)
assert all(chequeos.values())
print(f"\nUsados por el subconjunto: {int(sv.usado_en_modelo.sum())} de {len(sv)}")
print(f"Sin ningún caso en todo el dataset: {int((sv.casos_unicos == 0).sum())}")
print(f"Con casos, pero solo en enfermedades fuera del subconjunto: {int(((sv.usado_en_modelo == 0) & (sv.casos_unicos > 0)).sum())}")

In [ ]:
sv.sort_values("casos_subconjunto", ascending=False).head(10)[["nombre_en", "nombre_es", "casos_subconjunto"]]

**Decisión:** el vocabulario del modelo (`symptoms.json`) serán los síntomas con `usado_en_modelo = 1`. Los 49 síntomas sin ningún caso no aportan nada. Las traducciones son una primera versión y deben revisarse.

## 10. Conclusiones: qué debe hacer `transform.py`

| Hallazgo | Acción en `transform.py` |
|---|---|
| Nombres ya limpios, pero deben coincidir con la BD | aplicar `normalizar` a enfermedades y síntomas |
| 23% de filas duplicadas | `drop_duplicates()` **antes** de dividir train/val/test |
| 773 enfermedades, la mayoría con pocos casos | filtrar a las enfermedades de `label_map.csv` |
| Casos únicos ligados al número de síntomas | pesos por clase o división estratificada; no usar como prevalencia |
| 17% de filas con cuadro ambiguo | conservarlas; el modelo devuelve varias opciones con probabilidad |
| `regurgitation` y `regurgitation.1` | mantener ambas; la interfaz puede ocultar una |
| 49 síntomas sin casos y otros fuera del subconjunto | vocabulario = `usado_en_modelo == 1` (orden de `sintomas.csv`) |
| Sin edad, peso ni sexo en el dataset | `features.py` solo vectoriza síntomas |